# 使用 PROC SUMMARY 构建电信收入保障汇总立方体

## 执行摘要

某无线运营商的收入保障团队将一个月的用户日账单记录预先汇总为一个紧凑的汇总立方体，使分析师无需重新扫描原始事实表，即可按区域、套餐档位和通话类型深入钻取已结算收入。`PROC SUMMARY` 将 100 条用户日记录汇总为一组多维单元格：最细粒度（区域 x 套餐档位 x 通话类型）填充了 27 个可能单元格中的 25 个，而命名子立方体则给出了分析师最常查询的边际汇总。在本示例月份中，运营商在三个活跃区域共结算了 \$222.78，其中南区（\$97.44）和东区（\$86.94）合计占收入的 83%，北区以 \$38.40 垫底。收入最丰厚的单一子立方体是尊享版语音业务（18 条记录合计 \$59.06），而按每条记录产出对单元格排名则显示数据业务单元格是收入保障审计中价值最高的目标（最高产出 \$7.87/条记录）。以下每个数字均直接取自实际执行的输出。

## 数据来源

| 数据集 | 行数 | 粒度 | 关键变量 |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | 每行代表一条用户日使用汇总 | `region`（东区/南区/北区）、`plan_tier`（预付费/基础版/尊享版）、`call_type`（语音/短信/数据）、`device_class`、`bill_month`、`revenue`、`call_minutes`、`data_mb`、`subscriber_wt` |
| `work.cube_nway` | 25 | 每个非空的（region x plan_tier x call_type）单元格一行 | `_FREQ_`、`rev_total`、`rev_mean`、`rev_max`、`min_total`、`data_total` |
| `work.cube_hier` | 22 | 命名钻取子立方体的每个单元格一行 | `_TYPE_`、`_FREQ_`、`rev_total` |

所有数据均通过 `call streaminit()` / `rand()` 内联生成；不需要外部文件或网络访问。本环境运行于未授权模式，因此写入的数据集上限为 100 个观测值 —— 该场景的规模经过设计，使立方体能在此上限内被完整填充。

## 从原始通话详单到可钻取立方体

无线运营商需要跨数百万条每日使用事件结算收入。为了让收入保障分析师能够回答诸如 *“上个月南区尊享版语音收入是多少？”* 这样的问题，而无需每次都重新扫描原始事实表，我们将数据**预先汇总**为一个紧凑的汇总立方体。

`PROC SUMMARY` 正是 Base SAS 为此而生的主力工具：它按一个或多个 `CLASS` 维度对扁平事实表分组，并将请求的统计量写入输出数据集，同时为每一行标注 `_TYPE_`（哪些维度处于激活状态）和 `_FREQ_`（该单元格背后的记录数）。该输出数据集**就是**一个多维立方体 —— 与 OLAP 工具所暴露的汇总结构相同，只是以普通 SAS 数据集的形式实现，你可以打印、连接和切片它。

本 notebook 的流程：

1. 生成一个月真实的用户日账单记录。
2. 使用 `PROC SUMMARY NWAY` 构建最细粒度的立方体（区域 x 套餐档位 x 通话类型）。
3. 使用 `TYPES` 语句实现命名的**钻取子立方体**。
4. 使用 `WEIGHT` 将收入投影至用户基数，钻取到某个区域，并按每条记录产出对单元格排名以进行收入泄漏排查。

## 步骤 1 - 生成合成通话详单/账单数据

每行汇总了一位用户在某一天对某项服务的使用情况。我们使用 `call streaminit` 以确保可重现性，并使用 `rand()` 抽取合理的分布：收入与使用量随套餐档位而变化，语音收入与计费分钟数相关，数据收入与流量（MB）相关。每个 `RAND('table', ...)` 为每个类别分配一个概率，使 100 条记录的样本中每个区域、档位和通话类型都会出现。还附加了一个较小的 `subscriber_wt` 调查权重，供后续演示加权度量。

In [1]:
数据 work.cdr_billing;
    调用 streaminit(20260131);
    长度 region $12 plan_tier $15 call_type $12 device_class $15 bill_month $7;
    标签 revenue       = "已结算收入（美元）"
          call_minutes  = "计费通话分钟数"
          data_mb       = "数据流量（MB）"
          subscriber_wt = "用户调查权重";

    循环 i = 1 到 100;
        /* --- 维度：每个层级一个概率，总和为 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        选择 (r);
            当条件 (1) region = "东区";
            当条件 (2) region = "南区";
            其他 region = "北区";
        结束;

        p = rand("table", 0.30, 0.40, 0.30);
        选择 (p);
            当条件 (1) plan_tier = "预付费";
            当条件 (2) plan_tier = "基础版";
            其他 plan_tier = "尊享版";
        结束;

        c = rand("table", 0.50, 0.30, 0.20);
        选择 (c);
            当条件 (1) call_type = "语音";
            当条件 (2) call_type = "短信";
            其他 call_type = "数据";
        结束;

        如果 rand("uniform") < 0.55 那么 device_class = "智能机";
        否则 device_class = "功能机";

        bill_month = "2026-01";

        /* --- 度量，按档位与服务类型缩放 --- */
        tier_mult = (plan_tier = "预付费")*0.7
                  + (plan_tier = "基础版")*1.0
                  + (plan_tier = "尊享版")*1.7;

        call_minutes = round((call_type = "语音")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "数据")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "短信") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        输出;
    结束;
    删除 i r p c tier_mult base_rev;
运行;

过程 打印 数据=work.cdr_billing(obs=8) 标签 noobs;
    标题 "用户日账单记录样本";
运行;

                                                       用户日账单记录样本                                                        

region  plan_tier  call_type  device_class  bill_month                计费通话分钟数              数据流量（MB）                    已结算收入（美元）              用户调查权重
北区      尊享版        短信         智能机           2026-01                         0                     0                         0.67                1.13
南区      预付费        短信         功能机           2026-01                         0                     0                         0.94                1.42
南区      尊享版        短信         智能机           2026-01                         0                     0                         0.99                0.86
南区      尊享版        短信         智能机           2026-01                         0                     0                         1.01                1.03
南区      尊享版        语音         智能机           2026-01                     103.4                     0                         5.79     


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## 步骤 2 - 使用 PROC SUMMARY NWAY 构建最细粒度立方体

`NWAY` 只保留所有 `CLASS` 维度组合中最细粒度的那一种 —— 此处即每个已填充的（区域 x 套餐档位 x 通话类型）单元格。对每个单元格，我们存储收入的 `SUM`、`MEAN` 和 `MAX`，以及分钟数和流量总计，这样分析师就能读取每个单元格的收入总计、推算每条记录的平均值，并发现单笔最大的收费。`_FREQ_` 记录了每个单元格背后有多少用户日记录。此处我们丢弃 `_TYPE_`，因为在 NWAY 粒度下，每一行的类型都相同。

In [2]:
过程 summary 数据=work.cdr_billing NWAY;
    分类 region plan_tier call_type;
    变量 revenue call_minutes data_mb;
    输出 out=work.cube_nway(删除=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
运行;

过程 打印 数据=work.cube_nway(obs=12) 标签 noobs;
    标题 "NWAY 立方体单元格（区域 * 套餐档位 * 通话类型）";
    标签 region='区域' plan_tier='套餐档位' call_type='通话类型' _freq_='频数'
          rev_total='收入总计' rev_mean='收入均值' rev_max='最高收入'
          min_total='通话分钟总计' data_total='数据流量总计';
    格式 rev_total rev_mean rev_max min_total data_total comma10.2;
运行;

                                             NWAY 立方体单元格（区域 * 套餐档位 * 通话类型）                                              

    区域          套餐档位          通话类型      频数          收入总计          收入均值          最高收入              通话分钟总计              数据流量总计
东区      基础版           数据                 4         18.17          4.54          8.05                0.00            1,807.90
东区      基础版           短信                 4          4.07          1.02          1.24                0.00                0.00
东区      基础版           语音                 7         15.08          2.15          3.73              302.50                0.00
东区      尊享版           数据                 1          5.54          5.54          5.54                0.00              573.90
东区      尊享版           短信                 4          3.59          0.90          0.95                0.00                0.00
东区      尊享版           语音                 7         23.87          3.41          8.01              491.70                0.00
东区 


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## 步骤 3 - 使用 TYPES 实现命名的钻取子立方体

OLAP 立方体会预先存储分析师最常浏览的汇总层级。`TYPES` 语句正是做这件事：每一项都要求 `PROC SUMMARY` 生成一个子立方体。我们请求了总计 `()`、`region` 边际，以及两个二维子立方体 `region*plan_tier` 和 `call_type*plan_tier` —— 这些正是收入仪表盘会暴露的钻取路径。SAS 为每个子立方体标注一个 `_TYPE_` 代码（对 `CLASS` 列表的位掩码），因此单个输出数据集就能承载层级结构的每一级。

In [3]:
过程 summary 数据=work.cdr_billing;
    分类 region plan_tier call_type;
    变量 revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    输出 out=work.cube_hier
           sum(revenue)=rev_total;
运行;

过程 打印 数据=work.cube_hier 标签 noobs;
    标题 "子立方体汇总：总计、区域、区域*档位、通话类型*档位";
    变量 _type_ region plan_tier call_type _freq_ rev_total;
    标签 _type_='类型代码' region='区域' plan_tier='套餐档位'
          call_type='通话类型' _freq_='频数' rev_total='收入总计';
    格式 rev_total comma10.2;
运行;

                                               子立方体汇总：总计、区域、区域*档位、通话类型*档位                                               

        类型代码      区域          套餐档位          通话类型      频数          收入总计
           0                                         100        222.78
           3          基础版           数据                 8         38.06
           3          基础版           短信                 8          8.03
           3          基础版           语音                20         42.33
           3          尊享版           数据                 3         17.46
           3          尊享版           短信                13         11.75
           3          尊享版           语音                18         59.06
           3          预付费           数据                 7         14.50
           3          预付费           短信                 7          6.82
           3          预付费           语音                16         24.77
           4  东区                                      38         86.94
           4  北区          


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## 步骤 4 - 加权投影、区域钻取与泄漏排查

收入保障团队针对该立方体实际执行的三种查询：

- **加权投影。** 在 `region*plan_tier` 汇总中附加 `WEIGHT=subscriber_wt`，可报告按完整用户基数（而非原始采样行）缩放后的收入。
- **钻取。** 将 NWAY 立方体过滤到某一区域，可展开层级结构中的单一分支 —— 此处为南区 —— 展示其按档位与服务细分的详情。
- **泄漏排查。** 按每条记录平均收入对单元格排序，可发现产出最高的单元格；频次稀少但产出高的单元格正是审计要重点核查是否存在错误定价或收入泄漏的对象。

In [4]:
/* 加权收入，投影至用户基数 */
过程 summary 数据=work.cdr_billing NWAY;
    分类 region plan_tier;
    变量 revenue;
    权重 subscriber_wt;
    输出 out=work.cube_wt(删除=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
运行;

过程 打印 数据=work.cube_wt 标签 noobs;
    标题 "按区域 * 套餐档位加权收入（投影至用户基数）";
    标签 region='区域' plan_tier='套餐档位' rev_weighted='加权收入' records='记录数';
    格式 rev_weighted comma10.2;
运行;

/* 钻取南区分支 */
过程 打印 数据=work.cube_nway 标签 noobs;
    条件 region = "南区";
    标题 "深入南区：按档位与通话类型划分的收入单元格";
    变量 plan_tier call_type _freq_ rev_total rev_mean;
    标签 plan_tier='套餐档位' call_type='通话类型' _freq_='频数'
          rev_total='收入总计' rev_mean='收入均值';
    格式 rev_total rev_mean comma10.2;
运行;

/* 按每条记录产出对单元格排名以进行泄漏排查 */
过程 排序 数据=work.cube_nway out=work.cube_ranked;
    按照 DESCENDING rev_mean;
运行;

过程 打印 数据=work.cube_ranked(obs=6) 标签 noobs;
    标题 "平均收入最高的单元格（每条记录产出）";
    变量 region plan_tier call_type _freq_ rev_mean rev_max;
    标签 region='区域' plan_tier='套餐档位' call_type='通话类型'
          _freq_='频数' rev_mean='收入均值' rev_max='最高收入';
    格式 rev_mean rev_max comma10.2;
运行;

                                                按区域 * 套餐档位加权收入（投影至用户基数）                                                 

    区域          套餐档位          加权收入        记录数
东区      基础版                  50.85         15
东区      尊享版                  59.59         12
东区      预付费                  29.77         11
北区      基础版                  18.30          7
北区      尊享版                  22.42          7
北区      预付费                  20.56          9
南区      基础版                  58.63         14
南区      尊享版                  56.29         15
南区      预付费                  27.77         10

                                                 深入南区：按档位与通话类型划分的收入单元格                                                  

        套餐档位          通话类型      频数          收入总计          收入均值
基础版           数据                 3         12.02          4.01
基础版           短信                 2          2.01          1.00
基础版           语音                 9         22.51          2.50
尊享版           数据                 2         


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## 结果解读

汇总立方体将 100 行原始用户日数据转化为一组紧凑的预汇总单元格，可即时回答钻取类问题，而无需重新扫描事实表：

- **收入分布。** `region` 边际（`_TYPE_=4`）显示南区结算了 \$97.44，东区 \$86.94 —— 二者合计占当月 \$222.78 收入的 83%，北区贡献了 \$38.40。在 `call_type*plan_tier` 子立方体（`_TYPE_=3`）中，尊享版语音是收入最丰厚的单一单元格，18 条记录合计 \$59.06，其次是基础版语音的 \$42.33。
- **加权投影。** 应用调查权重后，排名会向用户权重更高的套餐倾斜：投影后的 `region*plan_tier` 收入中，东区-尊享版（\$59.59）和南区-基础版（\$58.63）领先 —— 这与未加权的区域总计呈现出不同的图景，提醒我们在评估风险敞口规模时应报告投影收入而非采样收入。
- **每条记录产出与泄漏排查。** 按 `rev_mean` 对 NWAY 单元格排名，数据业务单元格位居前列 —— 北区-基础版-数据（\$7.87/条记录）和南区-尊享版-数据（\$5.96/条记录）—— 这证实了高强度数据使用带来最高的单条记录收入。频次稀少的头部单元格（一两条记录）正是收入保障分析师会调取底层通话详单去核实高额收费是否被正确定价，而非出现错误的对象。

对于收入保障团队而言，这个立方体是方差仪表盘的基础：可以按（区域、档位、通话类型）单元格比较已结算收入与预期的费率表收入，偏差最大的单元格均值或加权总计即为值得审计的泄漏案例。由于整个结构都是普通的 SAS 数据集，下个月的立方体可以使用相同的 Base SAS 工具进行并集、差集或与费率表连接。